# Hafta 15 — Dönem Tekrarı ve Örnek Final Soruları

Bu defter, örnek final sorularını **elle** çözdükten sonra kontrol etmek içindir. Her hücre bir sorunun Python karşılığıdır; önce kâğıtta çözün, sonra çalıştırın.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
np.set_printoptions(precision=4, suppress=True)

## Soru 1 — Gradyan adımı (H5)

ŷ = wx + b; veri (2, 5), (4, 9); w = 1, b = 0, η = 0.05. Bir adım sonra w, b?

In [ ]:
x = np.array([2., 4.]); y = np.array([5., 9.]); w, b, eta = 1.0, 0.0, 0.05
yh = w*x + b; gw = (2/len(x))*np.sum((yh - y)*x); gb = (2/len(x))*np.sum(yh - y)
print("grad w =", gw, "grad b =", gb, "→ w =", w - eta*gw, "b =", b - eta*gb)
for olcek in (1, 100):
    ww, bb = 1.0, 0.0
    for i in range(5):
        yh = ww*(x*olcek) + bb; ww -= eta*(2/2)*np.sum((yh - y)*x*olcek); bb -= eta*(2/2)*np.sum(yh - y)
    print(f"x ölçeği {olcek}: 5 adım sonra w = {ww:.3g}  (ıraksama?)")

## Soru 2 — Maliyetli eşik (H6)

In [ ]:
for esik, TP, FP in ((0.5, 90, 40), (0.3, 130, 150)):
    FN = 150 - TP; P = TP/(TP + FP); R = TP/150; maliyet = FN*5000 + FP*300
    print(f"eşik {esik}: FN {FN}  P {P:.3f}  R {R:.3f}  F1 {2*P*R/(P+R):.3f}  maliyet {maliyet:,} ₺")
print("teorik eşik p* = C_FP/(C_FP+C_FN) =", round(300/5300, 3))

## Soru 3 — Geri yayılım (H10): elle ve autograd

In [ ]:
for w1 in (0.5, -0.5):
    w1t = torch.tensor(w1, requires_grad=True); w2t = torch.tensor(-2.0, requires_grad=True); x = torch.tensor(1.0); y = torch.tensor(1.0)
    z1 = w1t*x; a1 = torch.relu(z1); yh = w2t*a1; L = (yh - y)**2; L.backward()
    print(f"w1 = {w1}: L = {L.item():.1f}  dL/dw2 = {w2t.grad.item():.1f}  dL/dw1 = {w1t.grad.item():.1f}")

## Soru 4 — CNN boyutu ve parametre (H12)

In [ ]:
net = nn.Sequential(nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 4))
x = torch.zeros(1, 1, 64, 64)
for k in net: x = k(x); print(f"{k.__class__.__name__:18s} → {tuple(x.shape)}")
print("parametre:", sum(p.numel() for p in net.parameters()), " elle: 416 + 4640 + 132 =", 416 + 4640 + 132)
print("MAC ≈", 64*64*25*16 + 16*16*9*16*32, "→ ms @100 MHz:", round((64*64*25*16 + 16*16*9*16*32)/1e8*1e3, 1))

## Soru 5 — Zaman serisi senaryosu (H13): hatalı ve doğru hat, PV verisinde (24 saat ileri)

Hatalı hat: `rolling(24, center=True)` (geleceği içerir) + rastgele bölme + tüm veriyle scaler. Doğru hat: `shift(24).rolling(24)` + zaman bölmesi + eğitimle scaler.

**Dikkat:** Bu veride iki hat benzer MAE verir, çünkü `isinim` (o saatin ışınımı) özelliği üretimi neredeyse tek başına belirler ve sızıntının katkısı görünmez olur. Sızıntının hasarı, sızan bilginin ne kadar *yeni* olduğuna bağlıdır (4. haftadaki k-NN örneğinde R² 0.83 → −0.02). Hat yine de yanlıştır — ve asıl soru: 24 saat ileri tahminde `isinim(t)` gerçekten biliniyor mu? Hava tahmininden gelmiyorsa o da sızıntıdır.

In [ ]:
try:
    pv = pd.read_csv("pv_uretim.csv"); yv = pv.guc_kW.values; n = len(yv)
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    def ozellik(shift, merkez=False):
        X = pd.DataFrame({"lag24": pv.guc_kW.shift(24), "ort24": pv.guc_kW.shift(shift).rolling(24, center=merkez).mean(), "isinim": pv.isinim_Wm2, "sin": np.sin(2*np.pi*pv.saat/24)})   # senaryo: 24 s ileri tahmin, lag1 yok
        return X
    def mae(a, b): return np.mean(np.abs(a - b))
    # HATALI: rolling(24, center=True) hedefi ve 12 saat GELECEĞİ içerir, rastgele bölme, tüm veriyle scaler
    X = ozellik(0, merkez=True); ok = X.notna().all(axis=1).values; Xo, yo = X[ok].values, yv[ok]
    sc = StandardScaler().fit(Xo); a, b, ya, yb = train_test_split(sc.transform(Xo), yo, test_size=0.2, random_state=0)
    print("HATALI hat  MAE %.3f kW" % mae(yb, Ridge().fit(a, ya).predict(b)))
    # DOĞRU: shift(1), zaman bölmesi, scaler eğitimle
    X = ozellik(24); ok = X.notna().all(axis=1).values; idx = np.arange(n); ntr = int(0.8*n); tr = ok & (idx < ntr); ts = ok & (idx >= ntr)
    sc = StandardScaler().fit(X[tr]); pr = Ridge().fit(sc.transform(X[tr]), yv[tr]).predict(sc.transform(X[ts]))
    gunduz = yv[ts] > 0.5
    print("DOĞRU hat   MAE %.3f kW | naif 24 s MAE %.3f | gündüz MAPE (yanıltıcı) %.0f %% | gündüz nMAE %.1f %%" % (mae(yv[ts], pr), mae(yv[ts], yv[idx[ts] - 24]), 100*np.mean(np.abs(yv[ts][gunduz] - pr[gunduz])/yv[ts][gunduz]), 100*mae(yv[ts][gunduz], pr[gunduz])/10))
except FileNotFoundError: print("pv_uretim.csv yok — 5. haftanın verisini yükleyin.")

## Soru 6 — Kod okuma: çift softmax etkisi

In [ ]:
torch.manual_seed(0); z = torch.randn(64, 3)*3; y = torch.randint(0, 3, (64,))
print("doğru (logit → CE):        ", round(F.cross_entropy(z, y).item(), 3))
print("çift softmax (softmax → CE):", round(F.cross_entropy(torch.softmax(z, 1), y).item(), 3), " ← kayıp sıkışır, gradyan küçülür")

## Tekrar alıştırmaları (finale hazırlık)

1. **(H7)** 10 örnekli bir düğümde 6 A / 4 B var; bir bölme sol: 5 A / 1 B, sağ: 1 A / 3 B veriyor. Gini kazancını hesaplayın.
2. **(H8)** İki boyutlu 4 nokta: (0,0), (0,1), (5,0), (5,1); k = 2, başlangıç merkezleri (0,0) ve (5,1). k-means'i elle 1 iterasyon çalıştırın. Silhouette'i bir nokta için hesaplayın.
3. **(H11)** z = [2, 0, −1], y = 0 için softmax, CE ve ∂L/∂z.
4. **(H13)** 1440 saatlik seri, W = 48, h = 6, F = 3 için pencere sayısı ve tensör biçimi.
5. **(H14)** w = [−1.0, 0.0, 0.25, 0.5] için uint8 kuantizasyon (s, z, q) ve geri çevirme hatası.
6. **(H14)** Dört kutu: (200, 0.10, 0.12), (300, 0.30, 0.22), (300, 0.60, 0.58), (200, 0.90, 0.75) → ECE. Model nerede aşırı güvenli?